In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("crawford/20-newsgroups")

print("Path to dataset files:", path)


100%|██████████| 25.7M/25.7M [00:00<00:00, 155MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/crawford/20-newsgroups/versions/1


In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

#man in dataset ro az kaggle peyda krdm :))
def load_dataset(data_path):
    sentences = []
    labels = []
    for file_name in os.listdir(data_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(data_path, file_name)
            label = file_name.split(".")[0]
            with open(file_path, 'r', encoding='latin1') as file:
                text = file.readlines()
                sentences.extend(text)
                labels.extend([label] * len(text))
    return sentences, labels

#in kaht hm ta khat 30 khode kaggle gfte bud jaye download datset estefade knm
data_path = "/root/.cache/kagglehub/datasets/crawford/20-newsgroups/versions/1"
sentences, labels = load_dataset(data_path)

print(f"Number of sentences: {len(sentences)}")
print(f"Number of labels: {len(labels)}")

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
padded_sequences = pad_sequences(sequences, maxlen=100, padding='post')

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

train_X, test_X, train_Y, test_Y = train_test_split(
    np.array(padded_sequences), np.array(encoded_labels), test_size=0.2, random_state=42)

vocab_size = 5000
embedding_dim = 16
max_length = 100
num_classes = len(set(labels))

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    SimpleRNN(64),
    Dense(num_classes, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='nadam', metrics=['accuracy'])
model.fit(train_X, train_Y, epochs=10, validation_data=(test_X, test_Y), verbose=1)

def predict_topic(sentence):
    sequence = tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(sequence, maxlen=100, padding='post')
    prediction = model.predict(padded)
    predicted_label = label_encoder.inverse_transform([np.argmax(prediction)])
    return predicted_label[0]

new_sentence = "Blockchain is changing the financial world"
print("Predicted Topic:", predict_topic(new_sentence))

Number of sentences: 1719260
Number of labels: 1719260


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
42982/42982 ━━━━━━━━━━━━━━━━━━━━ 525s 12ms/step - accuracy: 0.2463 - loss: 1.7751 - val_accuracy: 0.3676 - val_loss: 1.5491
Epoch 2/10
42982/42982 ━━━━━━━━━━━━━━━━━━━━ 558s 12ms/step - accuracy: 0.3835 - loss: 1.5203 - val_accuracy: 0.3999 - val_loss: 1.4829
Epoch 3/10
42982/42982 ━━━━━━━━━━━━━━━━━━━━ 578s 12ms/step - accuracy: 0.4053 - loss: 1.4738 - val_accuracy: 0.4260 - val_loss: 1.4548
Epoch 4/10
42982/42982 ━━━━━━━━━━━━━━━━━━━━ 595s 13ms/step - accuracy: 0.4276 - loss: 1.4408 - val_accuracy: 0.4396 - val_loss: 1.4325
Epoch 5/10
42982/42982 ━━━━━━━━━━━━━━━━━━━━ 595s 13ms/step - accuracy: 0.4327 - loss: 1.4368 - val_accuracy: 0.4354 - val_loss: 1.4417
Epoch 6/10
42982/42982 ━━━━━━━━━━━━━━━━━━━━ 545s 13ms/step - accuracy: 0.4416 - loss: 1.4172 - val_accuracy: 0.4141 - val_loss: 1.4596
Epoch 7/10
42982/42982 ━━━━━━━━━━━━━━━━━━━━ 542s 13ms/step - accuracy: 0.4439 - loss: 1.4130 - val_accuracy: 0.4298 - val_loss: 1.4200
Epoch 8/10
42982/42982 ━━━━━━━━━━━━━━━━━━━━ 563s 13ms/s